In [2]:
import pandas as pd
import re

# 1. Cargamos el archivo .xlsx
# Usamos 'openpyxl' como motor para que lea el formato moderno de Excel
print("Cargando el archivo Excel .xlsx...")
df = pd.read_excel("datos_finales_limpios.xlsx", engine='openpyxl')

print(f"✅ Archivo cargado. Tenemos {len(df)} filas para analizar.")

# 2. Lista Maestra de Barrios (He añadido algunos más para asegurar el tiro)
barrios_vlc = [
    'Russafa', 'Ruzafa', 'Arrancapins', 'Benicalap', 'Benimaclet', 'Campanar', 
    'Cabañal', 'Cabanyal', 'Canyamelar', 'Malvarrosa', 'Patraix', 'Torrefiel', 
    'Ayora', 'Extramurs', 'Olivereta', 'Nou Moles', 'Tres Forques', 'Fuensanta', 
    'La Luz', 'Eixample', 'Gran Via', 'Montolivete', 'Monteolivete', 'En Corts', 
    'Malilla', 'La Punta', 'Ciutat Vella', 'El Carmen', 'El Carme', 'La Seu', 
    'El Pilar', 'La Xerea', 'Sant Francesc', 'Benimamet', 'Orriols', 'Mestalla', 
    'Exposicion', 'Algiros', 'Ciudad Jardin', 'Nazaret', 'Sant Pau', 'Penya-Roja',
    'Morvedre', 'Tavernes Blanques', 'Velluters', 'Saidia', 'Zaidia', 'Sant Isidre'
]

def extraer_barrio_final(row):
    enlace = str(row['Enlace']).lower()
    descripcion = str(row['Descripcion']).lower()
    
    # Lógica para Pisos.com
    if 'pisos.com' in enlace:
        slug = enlace.rstrip('/').split('/')[-1]
        match = re.search(r'(?:piso-|atico-|duplex-|casa-|chalet-)(.*?)(?=\d)', slug)
        if match:
            b = match.group(1).replace('_', ' ').replace('-', ' ').strip()
            return b.title()

    # Lógica para Idealista y otros (Búsqueda por texto)
    for barrio in barrios_vlc:
        if barrio.lower() in descripcion:
            return barrio.title()
            
    return "Valencia Capital (Zona General)"

# 3. Ejecutamos la magia
df['Barrio'] = df.apply(extraer_barrio_final, axis=1)

# 4. Unimos nombres repetidos
mapeo = {'Ruzafa': 'Russafa', 'El Carme': 'El Carmen', 'Cabanyal': 'Cabañal', 
         'Monteolivete': 'Montolivete', 'Gran Via': 'Gran Vía'}
df['Barrio'] = df['Barrio'].replace(mapeo)

print("-" * 50)
print(f"🏙️ ¡Zonificación lista! Barrios encontrados: {df['Barrio'].nunique()}")
print(df['Barrio'].value_counts().head(10))
print("-" * 50)



Cargando el archivo Excel .xlsx...
✅ Archivo cargado. Tenemos 1887 filas para analizar.
--------------------------------------------------
🏙️ ¡Zonificación lista! Barrios encontrados: 120
Barrio
Valencia Capital (Zona General)    141
Russafa                             85
El Cabanyal El Canyamelar           76
Sant Francesc                       56
Nou Moles                           54
Arrancapins                         53
Benicalap Barrio                    52
Aiora                               46
Montolivet                          44
Patraix Barrio                      43
Name: count, dtype: int64
--------------------------------------------------


In [ ]:
# Eliminamos específicamente las 3 columnas con exceso de nulos
columnas_a_quitar = ['Antiguedad', 'Exterior', 'Gastos_Comunidad']

# El parámetro errors='ignore' evita que el código de error si ya las habías borrado antes
df = df.drop(columns=columnas_a_quitar, errors='ignore')

print(" Columnas eliminadas con éxito.")
print(f"Columnas actuales en el dataset: {df.columns.tolist()}")

# 5. IMPORTANTE: Guardamos el RESULTADO en .csv para la IA
# El CSV es el formato "lengua franca" para los modelos de Machine Learning
df.to_csv('dataset_VALENCIA_ZONIFICADO.csv', index=False, sep=';', decimal=',', encoding='utf-8-sig')
print("💾 Guardado 'dataset_VALENCIA_ZONIFICADO.csv'. ¡Listo para entrenar la IA!")

✅ Columnas eliminadas con éxito.
Columnas actuales en el dataset: ['Enlace', 'Precio', 'Superficie_Construida', 'Superficie_Util', 'Habitaciones', 'Banos', 'Planta', 'Referencia', 'Descripcion', 'Ascensor', 'Garaje', 'Trastero', 'Terraza', 'Balcon', 'Jardin', 'Piscina', 'Aire_Acondicionado', 'Estado', 'Origen', 'Tipologia_Especial', 'Barrio']
💾 Guardado 'dataset_VALENCIA_ZONIFICADO.csv'. ¡Listo para entrenar la IA!
